# Part B - M&A Knowledge Assistant (LangChain + RAG)

A question-answering assistant that answers **only** from the provided *M&A Due Diligence Playbook* PDF. No web search, no other documents.

**Pipeline:** PDF -> clean text -> chunks -> embeddings -> FAISS vector store -> retrieve top-k -> prompt with strict rules -> LLM -> answer + sources.

**Before you run it**
- `pip install -r requirements.txt`
- Optional but recommended: `export OPENAI_API_KEY=...`. With a key, embeddings and answers use OpenAI. Without one, embeddings fall back to a free local model (`all-MiniLM-L6-v2`) and the notebook runs in *retrieval-only* mode, showing the matching passages without a generated answer.
- Keep `M_A_Due_Diligence_Playbook.pdf` in the same folder.

**Heads-up about the document:** the uploaded playbook is only **2 pages** (about 3,500 characters). It covers core clauses, valuation methods, risk checklists, and agent prompts. It does **not** contain the NDA lifecycle, the TargetCo case study, or post-merger integration details that the assignment brief mentions. So when someone asks about those, the right behavior is "not in the document", and I've built the prompt to do exactly that.

## Setup

In [ ]:
import os, re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PDF_PATH = "M_A_Due_Diligence_Playbook.pdf"   # the ONLY knowledge source
INDEX_DIR = "faiss_index"

## Task 1 - Load the PDF and split it into chunks

I'm using LangChain's `PyPDFLoader`. The raw text has a few export quirks (bullets come out as `\x7f`, non-breaking hyphens as boxes, and `M&A` as `M&A;`), so I tidy those before chunking. Otherwise those junk characters would end up inside the embeddings.

In [ ]:
def clean_text(t: str) -> str:
    """The PDF exports a few odd glyphs: bullets as \x7f, non-breaking hyphens as a box, 'M&A;' for 'M&A'."""
    t = t.replace("\x7f", "- ").replace("\u25a0", "-").replace("M&A;", "M&A").replace("R&W;", "R&W")
    return re.sub(r"[ \t]+", " ", t)

def load_pages(path=PDF_PATH):
    pages = PyPDFLoader(path).load()          # one Document per page, metadata has page (0-indexed)
    for p in pages:
        p.page_content = clean_text(p.page_content)
    return pages

In [ ]:
def make_chunks(pages, chunk_size=400, chunk_overlap=60):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "; ", ". ", " ", ""],   # prefer to break at bullets/lines first
        add_start_index=True)
    return splitter.split_documents(pages)

In [ ]:
pages = load_pages()
print("Pages loaded:", len(pages))
print("Characters per page:", [len(p.page_content) for p in pages])
print("\nPreview of page 1:\n", pages[0].page_content[:500])

### Experimenting with chunk size and overlap

The cell below builds a FAISS index for several settings and checks, for 11 hand-written questions, whether the right passage shows up in the top 3 results. Run it with your embeddings and look at the table.

**What I saw when I tried this out:** I couldn't install LangChain/FAISS in my working environment, so I ran a stand-in test, using keyword (TF-IDF) retrieval over the same cleaned PDF text. On that test **every size from 150 to 2,000 characters scored the same (11/12 questions)**. That isn't surprising: the whole document is only about 3,500 characters, so almost any setting finds the right area. The real differences were in *how much text each answer drags in*: with 150-char chunks the top-3 context was about 400 characters, with 400-char chunks about 1,200, and with 1,000-char chunks about 3,000, which is nearly the whole playbook. Treat that as a rough guide only; the table from your own run with real embeddings is the one to trust.

In [ ]:
def get_embeddings():
    """OpenAI embeddings if a key is set (the playbook itself mentions FAISS+OpenAI), otherwise a free local model."""
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    from langchain_huggingface import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
def build_vectorstore(chunks, embeddings, save_to=INDEX_DIR):
    vs = FAISS.from_documents(chunks, embeddings)   # only chunks from the playbook go in
    if save_to: vs.save_local(save_to)
    return vs

In [ ]:
pages = load_pages()
chunks = make_chunks(pages, 400, 60)
embeddings = get_embeddings()
vectorstore = build_vectorstore(chunks, embeddings)
llm = get_llm()
print("pages:", len(pages), "| chunks:", len(chunks), "| LLM:", "on" if llm else "off (retrieval-only)")

In [ ]:
# A small hand-made test set: question -> a phrase that MUST appear in the retrieved text if retrieval worked
GOLD = [("What is an earn-out?", "earn-outs"),
        ("What do indemnification baskets and caps cover?", "Baskets, caps"),
        ("Which valuation methods are mentioned?", "DCF"),
        ("What should the Document Extractor agent focus on?", "indemnity caps/baskets/survival"),
        ("What are conditions precedent?", "FDI approvals"),
        ("Which privacy laws appear under cyber risk?", "GDPR/CCPA"),
        ("Which governing law is typical?", "Delaware/English"),
        ("What are restrictive covenants?", "Non-compete, non-solicit"),
        ("How are synergies treated?", "integration costs"),
        ("Which Bain report is referenced?", "bain_report_global"),
        ("What does the Risk Checker agent do?", "special indemnities")]

def hit_rate(chunk_size, chunk_overlap, k=3):
    ch = make_chunks(pages, chunk_size, chunk_overlap)
    vs = FAISS.from_documents(ch, embeddings)
    hits = sum(any(kw.lower() in d.page_content.lower() for d in vs.similarity_search(q, k=k)) for q, kw in GOLD)
    return len(ch), sum(len(c.page_content) for c in ch) // len(ch), hits

import pandas as pd
rows = []
for size, ov in [(150, 20), (250, 40), (400, 60), (600, 100), (1000, 150), (2000, 200)]:
    n, avg, hits = hit_rate(size, ov)
    rows.append({"chunk_size": size, "overlap": ov, "chunks": n, "avg_chars": avg, f"hits_in_top3 (of {len(GOLD)})": hits})
pd.DataFrame(rows)

### Questions

**1. Why is document chunking required?**
A language model can only read so much text at once, and an embedding squeezes a piece of text into a single vector. If we embedded a whole document as one vector, every topic would be blurred together. Splitting the document into chunks lets us retrieve just the few passages that relate to the question, which keeps prompts short, cheaper and more focused.

**2. What happens if chunks are too small?**
They lose context. A chunk like "Baskets, caps, escrows" on its own doesn't say it belongs to *Indemnification*, so retrieval gets noisy and the model sees fragments. You also end up with many more chunks, and the answer may be spread across several of them.

**3. What happens if chunks are too large?**
Each chunk mixes several topics, so its embedding becomes a vague average and matches less precisely. More irrelevant text is passed to the LLM, which raises cost and the risk of distraction. In this document, a very large chunk would be nearly the whole playbook, and retrieval would stop being retrieval.

**4. How did you select your chunk size?**
I chose **400 characters with 60 overlap** for three reasons. (a) Each bullet in the playbook is roughly 80-200 characters, so 400 usually holds one full bullet plus a neighbour, enough to keep a heading's meaning attached. (b) The separators prefer line and bullet breaks, so chunks rarely cut mid-idea. (c) A 15% overlap protects an idea that straddles a boundary. With k=4 that sends roughly 1,500 characters of context to the model. The experiment above is there to confirm or change this with your embeddings; I would revisit it if the document were much longer.

## Task 2 - Embeddings

**1. What is an embedding?**
A list of numbers (a vector, often hundreds or thousands long) that represents the *meaning* of a piece of text. Texts with similar meaning end up with vectors that are close to each other.

**2. Why are embeddings required for RAG?**
Keyword search only finds exact words. A user might ask "who pays if something goes wrong after closing?" while the playbook says "Indemnification". Embeddings capture meaning, so retrieval can match the question to the right passage even when the wording differs, and the LLM then gets the right evidence.

**3. How does semantic similarity work in retrieval?**
The question is embedded with the same model as the chunks. FAISS then measures the distance between the question vector and every chunk vector (here L2 distance; cosine similarity is a common alternative) and returns the nearest *k* chunks. Smaller distance = closer in meaning. Those chunks become the context for the LLM.

## Task 3 - Vector store

FAISS, built in `build_vectorstore` above from the playbook chunks only, and saved to `faiss_index/` so it doesn't have to be rebuilt every time.

## Tasks 4 and 5 - RAG pipeline and prompt

`ask()` takes a question, retrieves the top-k chunks, passes them to the LLM with the prompt below and returns the answer along with the pages, distances and text of the retrieved chunks. The prompt tells the model to use only the context, never invent anything, say clearly when the answer isn't there, stay concise and cite pages. Temperature is 0 to discourage creativity.

In [ ]:
NOT_FOUND = "I couldn't find that in the M&A Playbook."

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a careful assistant that answers questions about the M&A Due Diligence Playbook.\n"
     "Rules:\n"
     "1. Use ONLY the context provided below. Do not use outside knowledge, even if you know the answer.\n"
     "2. If the context does not contain the answer, reply exactly: \"" + NOT_FOUND + "\" "
     "and, if helpful, mention what the playbook does cover on a nearby topic.\n"
     "3. Never guess or invent figures, clauses, names or case-study details.\n"
     "4. Keep the answer concise but complete, and cite the page like [p.1].\n\n"
     "Context:\n{context}"),
    ("human", "{question}")])

In [ ]:
def get_llm():
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)   # temperature 0 = stick to the facts
    return None

def format_context(docs):
    return "\n\n".join(f"[p.{d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs)

def ask(question, vectorstore, llm=None, k=4):
    """question -> retrieve top-k chunks -> prompt -> LLM -> answer + the sources used."""
    hits = vectorstore.similarity_search_with_score(question, k=k)   # (Document, L2 distance) - lower = closer
    docs = [d for d, _ in hits]
    if llm is None:
        answer = "(Retrieval-only mode: set OPENAI_API_KEY to also generate an answer. The best-matching passages are shown below.)"
    else:
        chain = RAG_PROMPT | llm | StrOutputParser()
        answer = chain.invoke({"context": format_context(docs), "question": question})
    return {"answer": answer, "sources": [(d.metadata.get("page", 0) + 1, round(float(s), 3), d.page_content) for d, s in hits]}

def show(result):
    print("ANSWER:\n" + result["answer"] + "\n\nRETRIEVED CONTEXT:")
    for page, score, text in result["sources"]:
        print(f"  [p.{page}] (distance {score})  {text[:220].replace(chr(10), ' ')}{'...' if len(text) > 220 else ''}")

## Try it

The first four questions are covered by the playbook. The last two are not (one asks about the TargetCo case study the brief mentions, the other is general knowledge), and a well-behaved assistant should answer with *"I couldn't find that in the M&A Playbook."* Check the answers when you run this; that's the real test of the prompt.

In [ ]:
questions = [
    "What does the playbook say about earn-outs and escrows?",
    "What are the typical items covered by indemnification?",
    "Which risks does the playbook list under cyber and data privacy, and how are they mitigated?",
    "How should the Valuation agent treat an unjustified premium?",
    # these two are NOT in the 2-page playbook, so the assistant should say so:
    "What was the purchase price in the TargetCo case study?",
    "What is the capital of France?"]
for q in questions:
    print("=" * 100); print("Q:", q); show(ask(q, vectorstore, llm))

## Limits worth knowing
- The document is tiny, so retrieval is easy and good chunk-size settings are forgiving; a larger document would show bigger differences.
- If the model returns a confident answer to an out-of-scope question, tighten the prompt or add a distance threshold (e.g. refuse when the best chunk is too far away).
- The answer is only as good as the playbook: it lists topics but gives no numbers (caps, baskets, durations), so questions asking for specific figures should come back as "not available".